[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/second_harmonic/second_harmonic.ipynb)

# A crystal that doubles the frequency

A strong light wave drives a crystal's charges nonlinearly, producing a polarization that oscillates twice as fast. Here the crystal is a bilinear extensor: two electric fields go in, one polarization comes out. We build it from four bonds, turn the crystal, bind one input, and follow the light from successive slices, flipping the crystal to keep it adding up.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.electromagnetism.second_harmonic import render

np.set_printoptions(precision=4, suppress=True)

# Electric fields and polarization are vectors of ordinary space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Rotor = ga.gatype.rotor()
Response = ga.gatype((Vector, Vector, Vector))                                # Vector <- (Vector, Vector)

# Light travels along a face diagonal of the crystal, with horizontal and vertical across it.
horizontal = (mv.y - mv.x) / np.sqrt(2)                                       # [] Vector
vertical = mv.z                                                                # [] Vector
propagation = (mv.x + mv.y) / np.sqrt(2)                                      # [] Vector
# The part of a polarization across the beam: only it radiates forward.
transverse = Vector - propagation * (propagation | Vector)                    # [] Vector <- Vector

## 1. Two electric fields in, one polarization out

A polar bond responds along its own direction to the product of the two fields along it. Four equal bonds pointing towards the corners of a tetrahedron give the symmetry of a zincblende crystal. A real pump `E * cos(t)` drives `response(E, E) * cos(t) ** 2`: half of it static, half oscillating at twice the frequency.

In Cartesian tensor notation the response reads as $P_i=\chi^{(2)}_{ijk}E_jE_k$, with the material scale absorbed into $\chi^{(2)}$.

In [ ]:
# Directed bonds: the opposite tetrahedron responds with the opposite sign.
bonds = mv.vector([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]) / np.sqrt(3)   # [bonds] Vector


def response(bonds: Vector) -> Response:
    """The polarization two fields drive: each bond responds along itself to the product of the
    two fields along it; each open Vector is its own input slot."""
    return 3 * np.sqrt(3) / 4 * (bonds * (bonds | Vector) * (bonds | Vector)).sum(axis=-1)   # [...] Vector <- (Vector, Vector)


def doubled(crystal: Response, pump: Vector) -> Vector:
    """The doubled-frequency polarization across the beam: half the response, the half that
    oscillates at twice the frequency."""
    return 0.5 * transverse(crystal(pump, pump))                              # [...] Vector


crystal = response(bonds)                                                     # [] Vector <- (Vector, Vector)
phase = np.linspace(0, 4 * np.pi, 401)
pump = horizontal * np.cos(phase)                                             # [times] Vector
# The polarization less its static half: the doubled-frequency oscillation.
harmonic = transverse(crystal(pump, pump)) - doubled(crystal, horizontal)     # [times] Vector

render.draw_waveform(phase, pump, harmonic);

## 2. Turn the crystal, keep the beam fixed

Turning the crystal changes both which field components drive its bonds and which way it responds. A rotor pulls both inputs back into the crystal and turns the output forward. The pale circle sweeps the pump's polarization, the orange curve and the polar plot show the doubled-frequency polarization and its power, and the arrows and dot mark the horizontal pump.

In Cartesian tensor notation turning the crystal reads as $\chi'_{ijk}=R_{ia}R_{jb}R_{kc}\chi_{abc}$.

In [ ]:
def turned(crystal: Response, rotation: Rotor) -> Response:
    """The response of the turned crystal. Turning only the output would leave the crystal's
    sensitivity fixed in the laboratory: both inputs must be pulled back too."""
    return rotation >> crystal(rotation << Vector, rotation << Vector)          # [...] Vector <- (Vector, Vector)


# Pump polarizations all around the beam, starting horizontal.
headings = np.linspace(0, 2 * np.pi, 361)
pumps = horizontal * np.cos(headings) + vertical * np.sin(headings)           # [pumps] Vector

render.draw_response(pumps, bonds, doubled(crystal, pumps));

In [ ]:
# Turn the crystal once about the beam, the pump fixed.
turns = np.linspace(0, 2 * np.pi, 72, endpoint=False)
rotations = ((horizontal ^ vertical) * (-turns / 2)).exp()                    # [frames] Rotor
frames = ((rotation >> bonds, doubled(turned(crystal, rotation), pumps)) for rotation in rotations)

movie = save_animation(render.animate(pumps, frames), "second_harmonic", 80)
display(Image(filename=str(movie)))

## 3. Fix one field and leave the other open

Binding a pump into one input leaves a linear map on the other. It is also how a weak additional field changes the doubled-frequency polarization: the two equal cross terms cancel the half. A circle of weak probe fields becomes an ellipse whose axes depend on the pump.

In differential notation, for $Q(E)=\tfrac12\chi(E,E)$, the first-order change is $DQ(E)[\delta E]=\chi(E,\delta E)$ when the two inputs are symmetric.

In [ ]:
fixed = stack((horizontal, (horizontal + vertical) / np.sqrt(2)))              # [cases] Vector
probes = 0.2 * pumps                                                          # [probes] Vector
# Bind the pump, keep the probe open.
probe_map = transverse(crystal(fixed[:, None], Vector))                       # [cases, 1] Vector <- Vector

render.draw_mixing(fixed, probes, probe_map(probes));

## 4. Keep the slices adding up

Each slice of the crystal launches doubled-frequency light, but the driving polarization and that light travel at different speeds, so later slices slip out of phase with earlier ones. The pseudoscalar carries the phase: it squares to minus one and commutes with every vector. With a slip of two full turns over the crystal a uniform crystal gives nothing back. Flipping the crystal every half turn of slip reverses each slice that would cancel, because inverting the bonds reverses the response: quasi-phase-matching.

In complex-amplitude notation the growth reads as $A(L)=\int_0^L s(z)\,e^{i\Delta k z}\,dz$, with $s=\pm1$ the orientation of the crystal at depth $z$.

In [ ]:
# Inverted bonds drive the opposite polarization, so a flipped slice adds with the opposite sign.
flipped = doubled(response(-bonds), horizontal) + doubled(crystal, horizontal)   # [] Vector

slices = 256
# The crystal in thin slices of equal thickness, one unit long in all.
thickness = 1 / slices
depths = (np.arange(slices) + 0.5) * thickness
# The slip over the crystal: none, two turns, and two turns flipped every quarter of the crystal.
mismatch = np.array([[0.0], [4 * np.pi], [4 * np.pi]])
orientation = np.stack([np.ones(slices), np.ones(slices), (-1.0) ** np.floor(depths * 4)])
slip = (mv.xyz * (mismatch * depths)).exp()                                   # [cases, slices] Phasor
# The light after each slice: the running sum of every slice's signed, slipped contribution.
amplitude = (slip * orientation * thickness).cumsum(axis=-1)                  # [cases, slices] Phasor

render.draw_growth(depths, amplitude);

In [ ]:
print("flipped bonds plus unflipped:", np.abs(flipped.kernel).max())
print("final power, matched, mismatched, flipped:", amplitude[:, -1].symmetric_reverse_product().to_array(), " (2/pi)^2:", (2 / np.pi) ** 2)

The bond construction follows the tetrahedral bulk model of [Hardhienata et al., *Bond Model and Group Theory of Second Harmonic Generation in GaAs(001)*](https://arxiv.org/abs/1408.1185). The propagation is the weak-conversion plane-wave limit of [MIT's *Optical Properties of Solids*, chapter 11](https://web.mit.edu/6.732/www/opt.pdf); flipping the crystal is how periodically poled frequency doublers are made. Intensities are normalized, not absolute conversion efficiencies.